# FTQC 中間レポート スターターNB: [[5,1,3]] 自作デコーダ

このノートブックは **足場(スターター)** です。完成解は入っていません。
syndrome の読み方を1ケースだけ手本として示し、**デコーダ本体(15エントリの表)とその応用はあなたが作ります**。

## 何が配布され、何を作るか

**配布(このNBに入っている / そのまま使ってよい)**
- [[5,1,3]] 符号の論理状態 |0_L> / |1_L> を準備する回路(完成形)
- スタビライザ生成元 g1-g4、論理演算子 X_L = XXXXX / Z_L = ZZZZZ
- syndrome の worked example を **1ケースだけ**(X1。残り14ケースはあなた)
- 2つの雑音モデル(符号容量 / CX)を別関数で分離したもの
- 実験ハーネスの骨(p を振って論理誤り率を集計しプロットする枠)

**あなたが作る(採点対象)**
- デコーダ: 15エントリの (syndrome -> recovery) 表
- 単一誤り訂正の実証: 任意の1量子ビットに X/Y/Z を注入 -> 訂正 -> 論理保持を15ケースで確認
- [A] 符号容量モデルでの break-even
- [S] CX 誤り抽出回路での利得変化の調査 + FT 必要性の考察

## 到達段階(冒頭に申告すること)

- **B(土台 / 単位ライン)**: デコーダ完成 + 単一 X/Y/Z 誤りの訂正実証
- **A(+加点B)**: 符号容量モデルで break-even(論理 おおよそ pの2乗 vs 非符号化 おおよそ p)
- **S(+加点C)**: CX 誤り抽出回路で利得の変化を調査 + FT 必要性の考察

提出する A4 まとめの冒頭に「どこまで実装したか(B / A / S)」を必ず明記してください。

## 考え方(CS のことばで)
- 誤り = 確率的な Pauli 演算子(X / Y / Z)
- syndrome = 各スタビライザと誤りの **交換関係**(可換 / 反可換)を読む2進パリティ
- 復号 = syndrome -> recovery の **lookup(表引き)**

物理のことば(粒子・波動関数など)は使いません。すべて線形代数とパリティで閉じます。


## 0. 環境とセットアップ

標的環境: Qiskit 2.4.2 / qiskit-aer 0.17.2 / STIM 1.16.0。バージョンを確認します。

In [ ]:
# 基本の import。numpy は syndrome 計算(交換関係)に、Qiskit は状態準備と回路に使う。
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Pauli, StabilizerState
from qiskit.synthesis import synth_circuit_from_stabilizers  # スタビライザから状態準備回路を合成する
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

import qiskit, qiskit_aer
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)
try:
    import stim
    print("stim", stim.__version__, "(任意セクションで使用)")
except Exception as e:
    print("stim 未導入(任意セクションのみ影響):", e)

## 1. 配布物: [[5,1,3]] 符号の定義

[[5,1,3]] は **非 CSS の完全符号**(距離3、5物理で1論理)。スタビライザは下の g1-g4。
論理演算子は X_L = XXXXX, Z_L = ZZZZZ。

非 CSS なので X 訂正と Z 訂正が分離せず、Steane 系(CSS)の復号パターンは移植できません。
そのため15通りの weight-1 Pauli(X/Y/Z 全部)を **1枚の syndrome 表に畳む** 必要があります。


In [ ]:
# スタビライザ生成元(配布値)。各文字列は左から qubit 0,1,2,3,4 上の Pauli。
G  = ['XZZXI', 'IXZZX', 'XIXZZ', 'ZXIXZ']   # g1, g2, g3, g4
XL = 'XXXXX'   # 論理 X
ZL = 'ZZZZZ'   # 論理 Z

# 完全符号の性質: 4スタビライザ -> 2^4 = 16 個の syndrome が
# 「誤りなし(I)」+「15個の weight-1 Pauli」と過不足なく1対1に対応する(衝突なし)。
# つまり recovery 表が一意に決まる。これが「15ケースで曖昧さゼロ」の根拠。
print("g1..g4 =", G)
print("X_L =", XL, " Z_L =", ZL)

In [ ]:
# 論理状態の準備回路(完成形・配布)。
# |0_L> は g1..g4 と Z_L のすべてを +1 固有値にする状態。
# synth_circuit_from_stabilizers にその5つを渡すと、準備回路が機械的に得られる(導出不要)。
prep0 = synth_circuit_from_stabilizers([*G, ZL])   # |0_L> を作る回路

# |1_L> は |0_L> に論理 X(= 各 qubit に X)を掛けたもの。
prep1 = prep0.copy()
for i, p in enumerate(XL):
    if p == 'X':
        prep1.x(i)

print("prep0 depth:", prep0.depth(), "ops:", dict(prep0.count_ops()))

# --- 自己検査: |0_L> が本当に g1..g4 と Z_L に +1 で固定されるか確認する ---
# (エンコーダが正しいことを実機で確かめる。あなたの実験の土台なので必ず通ること。)
ss0 = StabilizerState(prep0)
for s in [*G, ZL]:
    ev = np.real_if_close(ss0.expectation_value(Pauli(s)))
    assert abs(ev - 1) < 1e-9, f"{s} の固有値が +1 でない: {ev}"
# |1_L> では Z_L が -1 になるはず(論理ビットが反転している)
ss1 = StabilizerState(prep1)
zl_on_1 = np.real_if_close(ss1.expectation_value(Pauli(ZL)))
assert abs(zl_on_1 + 1) < 1e-9, f"|1_L> の Z_L が -1 でない: {zl_on_1}"
print("自己検査 OK: |0_L> は g1..g4,Z_L に +1、|1_L> は Z_L = -1")

## 2. syndrome worked example(1ケースのみ: X1)

**判定規則**: 誤り E とスタビライザ g が
- **反可換** なら syndrome ビット = **1**(g を測ると −1 が出る = 誤りが検出される)
- **可換** なら syndrome ビット = **0**

各 weight-1 誤りについて、g1-g4 それぞれとの可換/反可換を並べた4ビットが syndrome です。

### 例: X1(qubit 0 への X 誤り)

qubit 0 の上で、各生成元が持つ Pauli を見ると:

| 生成元 | qubit0 の Pauli | X との関係 | ビット |
|---|---|---|---|
| g1 = XZZXI | X | 可換(X と X) | 0 |
| g2 = IXZZX | I | 可換(X と I) | 0 |
| g3 = XIXZZ | X | 可換(X と X) | 0 |
| g4 = ZXIXZ | Z | 反可換(X と Z) | 1 |

X 誤りは qubit 0 にしか作用しないので、他の qubit の文字は無視してよい
(同じ qubit 上でのみ可換/反可換が問題になる)。よって

**s(X1) = (0, 0, 0, 1)**

下のセルでこの手順を実機でも確認します。


In [ ]:
# 交換関係の基本プリミティブ(配布・そのまま使ってよい)。
# 反可換 -> 1(syndrome ビットが立つ)、可換 -> 0、を返す。
def commutes_bit(pauli_a, pauli_b):
    """pauli_a と pauli_b が反可換なら 1、可換なら 0 を返す(syndrome ビットの定義そのもの)。"""
    return int(not Pauli(pauli_a).commutes(Pauli(pauli_b)))

# worked example: X1 の syndrome を g1..g4 に対して1つずつ求める。
X1 = 'XIIII'  # qubit 0 への X 誤り
s_X1 = tuple(commutes_bit(X1, g) for g in G)
print("s(X1) =", s_X1, " (手計算と一致するはず: (0, 0, 0, 1))")

## 3. 【あなたが作る①】デコーダ: 15エントリの (syndrome -> recovery) 表

worked example の手順(各 g との可換/反可換を4ビットに並べる)を、**残り14ケース**
(X2-X5, Y1-Y5, Z1-Z5)にも適用して、15個すべての syndrome を求めてください。

- 15個の syndrome は互いに相異なり、どれも (0,0,0,0) ではありません(完全符号の性質)。
- recovery は「その誤りを打ち消す Pauli」です。Pauli は自分自身が逆元(X·X=I)なので、
  weight-1 誤り E の recovery は E そのものになります。
- 完成した表(辞書など)から、syndrome を受け取って recovery を返す関数 `decode` を書いてください。

下に出発点として15個の誤りラベルを用意しました。`recovery_table` と `decode` を埋めてください。


In [ ]:
# 出発点: 扱うべき15個の weight-1 誤り(配布)。
def weight1(i, P):
    s = ['I'] * 5
    s[i] = P
    return ''.join(s)

ERRORS = [weight1(i, P) for i in range(5) for P in 'XYZ']   # X1,Y1,Z1,X2,... の15個
print("15個の誤り:", ERRORS)

# TODO(あなた): syndrome -> recovery の辞書を完成させる。
# ヒント: s = tuple(commutes_bit(E, g) for g in G) で各誤りの syndrome が出る。
recovery_table = {
    (0, 0, 0, 1): 'XIIII',   # 例(worked example の X1)。残り14個を同手順で追加する。
    # (?, ?, ?, ?): '....',
}

def decode(syndrome):
    """4ビット syndrome を受け取り recovery の Pauli 文字列を返す。
    syndrome が (0,0,0,0)(誤りなし)なら 'IIIII' を返すこと。"""
    # TODO(あなた): recovery_table を引いて recovery を返す実装に置き換える。
    raise NotImplementedError("decode を実装してください")

## 4. 【あなたが作る②】単一誤り訂正の実証(15ケース)

任意の1量子ビットに X / Y / Z を注入し、syndrome を求め、`decode` で recovery を引いて訂正し、
**論理状態が保持されること** を15ケースすべてで確認してください。

訂正の成否は **残差 residual = recovery · error** で判定できます(解析的に最軽量):
- residual が g1-g4 と可換、かつ X_L・Z_L とも可換 なら、論理状態は保たれた(訂正成功)。
- そうでなければ論理誤り(訂正失敗)。

下に Pauli の積 `pauli_mul`(配布)を用意しました。成否判定 `correction_ok` と15ケースのループは
あなたが書いてください。


In [ ]:
# Pauli 文字列どうしの積(位相は無視。残差が自明かどうかの判定にはこれで十分)。配布。
_MUL = {'XY':'Z','YX':'Z','XZ':'Y','ZX':'Y','YZ':'X','ZY':'X'}
def pauli_mul(a, b):
    """2つの Pauli 文字列の積を文字列で返す(位相は無視)。"""
    out = []
    for x, y in zip(a, b):
        if x == 'I':   out.append(y)
        elif y == 'I': out.append(x)
        elif x == y:   out.append('I')
        else:          out.append(_MUL[x + y])
    return ''.join(out)

# TODO(あなた): residual が論理状態を保つ(= 訂正成功)かを返す関数を書く。
# ヒント: commutes_bit(residual, g) が g1-g4 すべてで 0、かつ X_L,Z_L とも 0 なら成功。
def correction_ok(error, recovery):
    raise NotImplementedError("correction_ok を実装してください")

# TODO(あなた): 15個の誤りそれぞれについて
#   1) syndrome を計算  2) decode で recovery を引く  3) correction_ok で成否を確認
# し、15ケース全部が成功することを示す。

## 5. 2つの雑音モデル(配布・別関数で明示分離)

**混同しないこと。** 比較の baseline は常に「符号容量モデル」、加点C で足すのが「CX 誤り」です。

- `noise_codecapacity(p)`(加点B 用): 抽出はノイズレス。各物理量子ビットに per-qubit depolarizing を
  1回だけ与える。これが clean な おおよそ pの2乗 を生む。**「CX エラーレート」とは呼ばない**
  (B では CX に誤りを置かないため)。
- `noise_cx(p)`(加点C 用): syndrome 抽出を **実回路** にして、抽出に使う2量子ビットゲート
  (X 支持の cx と Z 支持の cz)に depolarizing を乗せる。

両モデルとも **測定はノイズレス**(readout error は入れない)。


In [ ]:
# --- 符号容量モデル(加点B): per-qubit depolarizing を1回サンプルする ---
# depolarizing: 各 qubit で確率 p のとき X/Y/Z を等確率で1つ適用、確率 1-p で I。
# 抽出はノイズレスなので、サンプルした Pauli 誤り E がそのまま「真の誤り」になる。
_rng = np.random.default_rng()
def noise_codecapacity(p):
    """符号容量 / per-qubit depolarizing。5物理に独立に Pauli 誤りを置き、誤り文字列を返す。"""
    out = ['I'] * 5
    for i in range(5):
        if _rng.random() < p:
            out[i] = _rng.choice(['X', 'Y', 'Z'])
    return ''.join(out)

# 動作確認(誤りのサンプルが出るだけ。決め打ちの数値ではない)
print("符号容量サンプル例(p=0.2):", [noise_codecapacity(0.2) for _ in range(5)])

In [ ]:
# --- CX 誤りモデル(加点C): 実回路の2量子ビットゲートに depolarizing を乗せる NoiseModel ---
def noise_cx(p):
    """抽出回路の2量子ビットゲート(cx, cz)に 2量子ビット depolarizing(p)を付与する。
    非 CSS の重み4スタビライザは X 支持を cx、Z 支持を cz で測るため、両方に雑音を乗せないと
    Z 支持ゲートが無雑音になり加点C の雑音を過小評価する。測定はノイズレス(readout error なし)。"""
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx', 'cz'])
    return nm

# 重要な落とし穴: transpiler が抽出回路を最適化すると cx/cz が消えて雑音の置き場がずれる。
# optimization_level=0(または barrier)で抽出回路の構造を保持すること(#11 で既知の系と同型)。
print("noise_cx(0.05) 構築OK:", noise_cx(0.05).noise_instructions)

## 6. 実験ハーネス(骨・配布)

p を振って、符号化版(あなたの `decode` を通す)と非符号化版の論理誤り率を集計します。
**デコーダと成否判定はあなたの実装(セル3-4)を使います。** 下の枠はそのまま再利用してよい部分です。

非符号化 baseline は「1物理量子ビットに同じ depolarizing p を与えたメモリ比較」とします
(論理操作を伴わない素直な比較)。距離3 -> 1誤り訂正 -> 論理失敗には2誤り必要 -> 論理誤り
おおよそ pの2乗、という対応がプロットに現れるはずです。


In [ ]:
def logical_error_rate_codecapacity(p, shots=20000):
    """符号容量モデルでの符号化版・論理誤り率を Monte Carlo で推定(あなたの decode を使う)。"""
    fails = 0
    for _ in range(shots):
        e = noise_codecapacity(p)                 # 真の誤り(Pauli)
        s = tuple(commutes_bit(e, g) for g in G)  # syndrome を読む
        r = decode(s)                              # あなたのデコーダ
        if not correction_ok(e, r):                # あなたの成否判定
            fails += 1
    return fails / shots

def unencoded_error_rate(p):
    """非符号化(1物理)メモリ比較の baseline。
    1量子ビットが depolarizing p で「壊れる」確率 = p(誤り I 以外が乗る確率)。"""
    # TODO(あなた): 公平比較の定義を確認し、必要なら見直す(設計上はメモリ比較 = p)。
    return p

# 実行は decode / correction_ok を実装してから。
P_LIST = [0.02, 0.04, 0.06, 0.08, 0.10, 0.13, 0.16, 0.20]
# encoded = [logical_error_rate_codecapacity(p) for p in P_LIST]
# unenc   = [unencoded_error_rate(p) for p in P_LIST]

In [ ]:
# プロット枠(配布・再利用部)。encoded / unenc を作ってから実行する。
import matplotlib.pyplot as plt
def plot_breakeven(p_list, encoded, unenc):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(p_list, unenc,   'o-', label='unencoded (~ p)')
    ax.plot(p_list, encoded, 's-', label='encoded [[5,1,3]] (~ p^2)')
    ax.plot(p_list, p_list, 'k:', alpha=0.4, label='y = p (guide)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('physical error rate p'); ax.set_ylabel('logical error rate')
    ax.set_title('code-capacity break-even ([[5,1,3]])')
    ax.legend(); ax.grid(True, which='both', alpha=0.3)
    return fig
# 使い方:
# fig = plot_breakeven(P_LIST, encoded, unenc); plt.show()
print("plot_breakeven 準備OK(encoded/unenc を作ってから呼ぶ)")

## 7. 【加点C】CX 誤り抽出回路(実回路)の骨

加点C では syndrome 抽出を **実回路** で行います。weight-4 のスタビライザの抽出には2量子ビットゲート
(X 支持は cx、非 CSS ゆえ Z 支持は cz)が多数必要で、ここに `noise_cx(p)` で誤りを乗せます
(cx と cz の両方。片方だけだと雑音を過小評価します)。物理規模はアンシラを足して 6-9 物理
(密度行列でも 4^9 で軽い)。

下は1つのスタビライザ(g1)の抽出回路の最小テンプレートです。**全スタビライザの抽出 +
syndrome からの復号 + 集計** はあなたが組み立ててください。比較は「加点B の符号容量を baseline に固定し、
そこに CX 誤りを足したら baseline からどう動くか」の差分として1本の軸で見ます。


In [ ]:
def extract_one_stabilizer(g, data=range(5), ancilla=5):
    """1つのスタビライザ g をアンシラ1本で抽出する最小回路(g の各 Pauli を制御演算で写す)。
    X 成分は CX、Z 成分は CZ、Y 成分は両方、で測る(ここでは構造の最小例として X/Z を扱う)。"""
    qc = QuantumCircuit(6, 1)
    qc.h(ancilla)                       # アンシラを |+> にしてパリティを書き込む
    for i, p in enumerate(g):
        if p == 'X':   qc.cx(ancilla, i)
        elif p == 'Z': qc.cz(ancilla, i)
        elif p == 'Y': qc.cx(ancilla, i); qc.cz(ancilla, i)
    qc.barrier()                        # 抽出構造を最適化から守る(opt_level=0 と併用)
    qc.h(ancilla); qc.measure(ancilla, 0)
    return qc

# 雑音下で1回回す煙テスト(opt_level=0 で CX 構造を保持)。
sim = AerSimulator(method='density_matrix')
demo = transpile(extract_one_stabilizer(G[0]), sim, optimization_level=0)
res = sim.run(demo, noise_model=noise_cx(0.05), shots=2000).result()
ops = dict(demo.count_ops())
print("g1 抽出の煙テスト counts:", res.get_counts(), "| 保持された cx:", ops.get('cx'), "cz:", ops.get('cz'))

## 8. (任意)STIM 最小例

符号容量モデルは Clifford + Pauli 雑音なので STIM でも素直に回ります(大きな p sweep を速く回したいとき向け)。
基本は Qiskit のままで構いません。下は STIM で5量子ビットの状態にスタビライザ測定をかける最小例です。


In [ ]:
try:
    import stim
    sim = stim.TableauSimulator()
    # 何らかの状態を一例として作り、g1 = XZZXI のパリティを1回測る最小デモ(API の形の紹介)。
    sim.h(0)
    g1 = stim.PauliString("XZZXI")
    val = sim.measure_observable(g1)
    print("STIM で g1=XZZXI のパリティを測った例(真偽値):", val)
    print("符号容量の大規模 sweep を STIM で組むのは任意。基本は Qiskit のままでよい。")
except Exception as e:
    print("STIM 未導入のためスキップ:", e)

## 9. 物理規模メモ + 提出チェックリスト

**物理規模**
- 符号容量(解析 syndrome): 5物理。
- 実回路抽出(加点C): + アンシラで 6-9 物理。密度行列でも 4^9 で軽い。
- 前回の上限(<=15物理)に遠く及ばず、ローカルで十分回る。

**提出物**
- A4 1-2枚のまとめ(到達段階・結果・考察)+ 実装NB(.ipynb)。
- まとめ冒頭に **到達段階(B / A / S)** を必ず明記。

**到達段階の目安**
- B: デコーダ(15表)完成 + 単一誤り訂正を15ケースで実証。
- A: + 符号容量モデルで break-even を同定し、距離3 -> pの2乗 を説明。
- S: + CX 誤り下で利得変化を調査し、FT 抽出・より大きな符号の必要性と結びつけて考察。
  (利得が消える/残るの結論そのものではなく、考察の正しさを評価。)
